Load our dataset and prepare minibatches

In [ ]:
import torch as Torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt

#normalize our data to a bunch of -1 to +1 tensors
normalize_transforms = torchvision.transforms.Compose(
    [torchvision.transforms.ToTensor(),
     torchvision.transforms.Normalize(mean=(0.5), std=(0.5))]
)

trainData = torchvision.datasets.MNIST(root='MNIST/train', train=True, transform=normalize_transforms, download=True)
testData = torchvision.datasets.MNIST(root='MNIST/test',train=False, transform=normalize_transforms,download=True)

print("We have loaded our datasets")

#make batches of our data
batchSize = 128
trainDataLoader = Torch.utils.data.DataLoader(
    trainData, batch_size=batchSize
)
testDataLoader = Torch.utils.data.DataLoader(
    testData, batch_size=batchSize
)



Visualize and Analyze our data

In [ ]:
print("*****\nPREPARING MINIBATCHES")
#prepare minibatches
batch_size = 128
train_loader = Torch.utils.data.DataLoader(
    trainData, batch_size=batch_size)
test_loader = Torch.utils.data.DataLoader(testData, batch_size=batch_size)

#visualization
print("******\nVISUALIZE DATA SET")
dataiter = iter(train_loader)
images, labels = next(dataiter)
plt.imshow(np.transpose(torchvision.utils.make_grid(
    images[:25], normalize=True, padding=1, nrow=5).numpy(), (1, 2, 0)))
plt.axis('off')
plt.show()



#analyze dataset
print("*******\nANALYZE DATASET")
classes = []
for batch_idx, data in enumerate(train_loader):
    x, y = data
    classes.extend(y.tolist())

unique, counts = np.unique(classes, return_counts=True)
names = list(testData.class_to_idx.keys())
plt.bar(names, counts)
plt.xlabel("Target Classes")
plt.ylabel("Number of training instances")
plt.show()


Build our model! yay!

In [ ]:
class CNN(Torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = Torch.nn.Sequential( #building layers
            Torch.nn.Conv2d(in_channels=1, out_channels=32,kernel_size=3, padding=1), #1 input channels bcus greyscale channels for image, kernel size of 3 is a 3x3 filter
            Torch.nn.ReLU(),
            Torch.nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1), 
            Torch.nn.ReLU(),
            Torch.nn.MaxPool2d(kernel_size=2),
            Torch.nn.Conv2d(in_channels=64, out_channels=48, kernel_size=3, padding=1), 
            Torch.nn.ReLU(),
            Torch.nn.MaxPool2d(kernel_size=2),
            Torch.nn.Conv2d(in_channels=48, out_channels=64, kernel_size=3, padding=1), 
            Torch.nn.ReLU(),
            Torch.nn.Flatten(), #flatten layers now that we're done
            Torch.nn.Linear(64*7*7, 300), #fully connected flatten, 
            Torch.nn.ReLU(),
            Torch.nn.Linear(300, 10)
            
        )

    def forward(self, x):
        return self.model(x)
    


In [ ]:
device = 'cpu'
if Torch.cuda.is_available():
    device='cuda'

print(device)
model = CNN().to(device)

Train Our model! So Fun!

In [ ]:
num_epochs = 5
learning_rate = 0.000776
weight_decay = 0.00776
criterion = Torch.nn.CrossEntropyLoss()
optimizer = Torch.optim.Adam(
    model.parameters(), lr=learning_rate, weight_decay=weight_decay)


#train
print("\n\n*****************\nTRAINING STARTING\n****************")
train_loss_list = []
for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}:', end=' ')
    train_loss = 0
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss_list.append(train_loss / len(train_loader))
    print(f"Training loss = {train_loss_list[-1]}")
print("\n\n*****************\nTRAINING DONE\n****************")
plt.plot(range(1, num_epochs + 1), train_loss_list)
plt.xlabel("Number of epochs")
plt.ylabel("Training loss")
plt.show()

print(f"Final loss: {train_loss_list[-1]}")

Test the model! This is necessary!

In [24]:
print("****************\nTESTING STARTING\n******************")
test_acc = 0
model.eval()
with Torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        y_true = labels.to(device)
        outputs = model(images)
        _, y_pred = Torch.max(outputs.data, 1)
        test_acc += (y_pred == y_true).sum().item()

print(f"Test set accuracy = {100 * test_acc / len(testData)} %")
num_images = 10
y_true_name = [names[y_true[idx]] for idx in range(num_images)]
y_pred_name = [names[y_pred[idx]] for idx in range(num_images)]
title = f"Actual labels: {y_true_name}, Predicted labels: {y_pred_name}"

plt.imshow(np.transpose(torchvision.utils.make_grid(
    images[:num_images].cpu(), normalize=True, padding=1).numpy(), (1, 2, 0)))
plt.title(title)
plt.axis("off")
plt.show()


In [26]:
#save the model
Torch.save(model.state_dict(), "./drive/MyDrive/CYBR486/models/Naisu.pt")

In [ ]:
#Load the model
model.load_state_dict(
    Torch.load("./drive/MyDrive/CYBR486/models/9833Test-8947Dev.pt", map_location="cpu")
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')